In [12]:
import matplotlib.pyplot as plt
import nelpy as nel
import nelpy.plotting as npl
import numpy as np
import seaborn as sns
from neuro_py.io import loading
from neuro_py.plotting.figure_helpers import set_size
from neuro_py.tuning import maps
from skimage import measure
import os
import pandas as pd
import scipy.io as sio
import logging

logging.getLogger().setLevel(logging.ERROR)
basepath =  r"S:\data\latentseq\RM10\rm10_day11_20250815"
basename=os.path.basename(basepath)# Define the new folder path
#create working folder
new_folder_path = os.path.join(basepath, "complex_system")
brain_region="CA1"
animal="Rat"   #or Mouse
cell_type='all'
path_parts = basepath.split("\\")  # Split the path using backslash
animal_name = path_parts[-2]  # Second-to-last part is the animal name
day = path_parts[-1]  # Last part is the day
save_dir = r"D:\Documents\GitHub\R_Project\PPP\codebook\Criticality_analysis\complex_system\manifold"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
# Create the folder if it doesn't exist
if not os.path.exists(new_folder_path):
    os.makedirs(new_folder_path)
    print(f"Folder created: {new_folder_path}")
else:
    print(f"Folder already exists: {new_folder_path}")
# load in spike data from hpc pyramidal cells
st, cm = loading.load_spikes(
    basepath, brainRegion=brain_region                          # putativeCellType="Pyr"
)
# load theta epochs
state_dict = loading.load_SleepState_states(basepath)
nrem_epochs = nel.EpochArray(
    state_dict["NREMstate"],
)
awake_epochs = nel.EpochArray(
    state_dict["WAKEstate"],
)
# theta_epochs = nel.EpochArray(
#     state_dict["THETA"],
# )
rem_epochs=nel.EpochArray(state_dict["REMstate"],)
try:
    # Try default loader
    epoch_df = loading.load_epoch(basepath)
except Exception as e:
    print(f"load_epoch failed with error: {e}")
    print("Falling back to manual .mat file load...")

    # Load the .mat file
    mat = sio.loadmat(os.path.join(basepath, "DayNight25.MergePoints.events.mat"), squeeze_me=True)

    # Extract the 'MergePoints' structure
    merge_points = mat['MergePoints']

    # Extract the timestamps array: shape (N, 2)
    timestamps = merge_points['timestamps']
    # Unwrap the array inside
    timestamps = np.array(timestamps.tolist())

    epoch_df = pd.DataFrame(timestamps, columns=["startTime", "stopTime"])

sleep_epochs = nel.EpochArray(np.array([epoch_df.startTime, epoch_df.stopTime]).T)
E_spikes = st.time[(cm.putativeCellType == 'Pyramidal Cell')]
I_spikes= st.time[(cm.putativeCellType.str.contains("Interneuron"))]
spike_times=st.time
n_cell=len(spike_times)
print(f"number of cells: {n_cell}")

Folder already exists: W:\data\PPP\PPP20\day5\complex_system
number of cells: 297


In [13]:
import os, re, numpy as np, pandas as pd
import glob
def _load_mat_any(mat_path):
    from scipy.io import loadmat
    try:
        return loadmat(
            mat_path,
            squeeze_me=True,
            struct_as_record=False,
            simplify_cells=True
        )
    except TypeError:
        return loadmat(
            mat_path,
            squeeze_me=True,
            struct_as_record=False
        )
    except NotImplementedError:
        import mat73
        return mat73.loadmat(mat_path)
def _to_1d_array(x):
    """
    Coerce a MATLAB-loaded field to a 1D numpy array.
    - Scalars -> 1-length array
    - Lists/tuples/ndarrays -> flattened 1D
    - None / missing -> empty array
    """
    if x is None:
        return np.array([], dtype=float)
    arr = np.asarray(x, dtype=object)  # start as object to avoid surprises
    if arr.ndim == 0:
        # scalar or unsized object
        return np.atleast_1d(arr).astype(float, copy=False)
    # flatten and try float cast (ignore non-numeric gracefully)
    try:
        return np.asarray(arr).ravel().astype(float, copy=False)
    except (ValueError, TypeError):
        # fallback: best-effort, convert elementwise
        return np.array([float(v) for v in np.ravel(arr)], dtype=float)

def _to_state_str(s):
    # MATLAB char arrays sometimes come in as numpy arrays
    try:
        if isinstance(s, (bytes, bytearray)):
            return s.decode('utf-8', 'ignore')
        if isinstance(s, str):
            return s
        # arrays or weird containers -> join characters
        arr = np.asarray(s)
        if arr.ndim == 0:
            return str(arr.item())
        # e.g., array of characters
        return ''.join(arr.tolist()) if arr.dtype.kind in ('U','S') else str(arr)
    except Exception:
        return str(s)

def load_d2_timeline_complete(basepath, region):
    """Complete version that loads ALL sleep-state data, including d2_err."""
    region = region.upper()
    search_dir = os.path.join(basepath, "complex_system")
    # pats = [f"*_{region}_D2_time_resolved_rippleablate.mat", f"*_{region}_D2_time_rippleablate.mat"]
    pats = [f"*_{region}_D2_time_resolvedforplot.mat", f"*_{region}_D2_timeforplot.mat"]
    files = []
    for pat in pats:
        files.extend(glob.glob(os.path.join(search_dir, pat)))
    if not files:
        raise FileNotFoundError(f"No *_{region}_D2_time*.mat under {search_dir}")

    mat_path = max(files, key=os.path.getmtime)
    print(f"Loading: {os.path.basename(mat_path)}")

    mat = _load_mat_any(mat_path)
    results = mat['results']
    params = mat.get('params', {})

    # Make results iterable no matter if it's a list, ndarray, or dict
    if isinstance(results, dict):
        iter_results = list(results.values())
    else:
        iter_results = list(results if isinstance(results, (list, tuple)) else np.atleast_1d(results))

    print(f"📊 Found {len(iter_results)} result items (sleep states)")
    print(f"📊 Parameters: {params}")

    # Try to infer animal/day from filename if available
    fname = os.path.basename(mat_path)
    animal_name = None
    day = None
    m_day = re.search(r'day(\d+)', fname, flags=re.I)
    if m_day:
        day = int(m_day.group(1))
    m_animal = re.match(r'([^_]+)_day\d+_', fname)
    if m_animal:
        animal_name = m_animal.group(1)

    all_centers, all_d2, all_d2err, all_states = [], [], [], []
    n_states_with_err = 0

    for i, result_item in enumerate(iter_results):
        # Access as dict-like regardless of MATLAB struct quirks
        state = _to_state_str(result_item.get('state', 'UNKNOWN')) if hasattr(result_item, 'get') else \
                _to_state_str(result_item['state']) if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) else 'UNKNOWN'

        centers_raw = result_item.get('center', []) if hasattr(result_item, 'get') else \
                      (result_item['center'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'center' in result_item else [])
        d2_raw = result_item.get('d2', []) if hasattr(result_item, 'get') else \
                 (result_item['d2'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'd2' in result_item else [])
        d2err_raw = result_item.get('d2_err', []) if hasattr(result_item, 'get') else \
                    (result_item['d2_err'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'd2_err' in result_item else [])

        centers  = _to_1d_array(centers_raw)
        d2_vals  = _to_1d_array(d2_raw)
        d2_err   = _to_1d_array(d2err_raw)

        # Presence flag (some states may not have d2_err)
        has_err = d2_err.size > 0 and np.any(np.isfinite(d2_err))
        n_states_with_err += int(has_err)

        # Harmonize lengths: truncate all to the common min
        sizes = [centers.size, d2_vals.size] + ([d2_err.size] if has_err else [])
        n_pts = min(sizes) if min(sizes) > 0 else 0
        print(f"   State {i}: {state} - requested points={centers.size}, actual used={n_pts}")

        if n_pts == 0:
            print("     ⚠️ Empty state - no data points")
            continue

        centers = centers[:n_pts]
        d2_vals = d2_vals[:n_pts]
        d2_err  = (d2_err[:n_pts] if has_err else np.full(n_pts, np.nan, dtype=float))

        # Ranges
        try:
            print(f"     Time range: {float(np.nanmin(centers))} to {float(np.nanmax(centers))}")
        except ValueError:
            print("     Time range: (all NaN)")

        try:
            print(f"     D2 range: {float(np.nanmin(d2_vals)):.4f} to {float(np.nanmax(d2_vals)):.4f}")
        except ValueError:
            print("     D2 range: (all NaN)")

        if np.any(np.isfinite(d2_err)):
            try:
                dmin, dmax = float(np.nanmin(d2_err)), float(np.nanmax(d2_err))
                print(f"     D2_ERR range: {dmin:.4f} to {dmax:.4f}")
            except ValueError:
                print("     D2_ERR range: (all NaN)")
        else:
            print("     D2_ERR: missing → filled with NaN")

        all_centers.append(centers)
        all_d2.append(d2_vals)
        all_d2err.append(d2_err)
        all_states.append(np.array([state]*n_pts, dtype=object))

    if len(all_centers) == 0:
        raise ValueError("No data found in any sleep state - all states are empty")

    # Concatenate
    all_centers = np.concatenate(all_centers)
    all_d2      = np.concatenate(all_d2)
    all_d2err   = np.concatenate(all_d2err)
    all_states  = np.concatenate(all_states)

    # DataFrame (include d2_err)
    df = pd.DataFrame({
        'center_s': all_centers.astype(float, copy=False),
        'd2':       all_d2.astype(float, copy=False),
        'd2_err':   all_d2err.astype(float, copy=False),
        'state':    all_states.astype(str)
    }).sort_values('center_s').reset_index(drop=True)

    print(f"\n✅ Complete timeline created:")
    print(f"📊 Total data points: {len(df)}")
    print(f"📊 Time range: {df['center_s'].min()} to {df['center_s'].max()}")
    print(f"📊 Duration: {(df['center_s'].max() - df['center_s'].min()):.1f} seconds")
    print(f"📊 State distribution:")
    print(df['state'].value_counts())

    # Empty-state tracker
    empty_states = []
    for ri in iter_results:
        st = _to_state_str(ri.get('state','UNKNOWN')) if hasattr(ri,'get') else \
             _to_state_str(ri['state']) if (isinstance(ri,dict) or hasattr(ri,'__getitem__')) and 'state' in ri else 'UNKNOWN'
        csz = np.size(np.atleast_1d(ri.get('center', []))) if hasattr(ri,'get') else \
              np.size(np.atleast_1d(ri['center'])) if (isinstance(ri,dict) or hasattr(ri,'__getitem__')) and 'center' in ri else 0
        if csz == 0:
            empty_states.append(st)

    meta = {
        "animal_name": animal_name,
        "day": day,
        "region": region,
        "path": mat_path,
        "params": params,
        "columns": list(df.columns),
        "shape": df.shape,
        "n_states": len(iter_results),
        "states_with_d2_err": n_states_with_err,
        "empty_states": empty_states
    }

    return df, meta



# Load the CA1 d2 timeline
try:
    ca1_complete_df, ca1_complete_meta = load_d2_timeline_complete(basepath, "CA1")
    
    # pfc_complete_df, pfc_complete_meta = load_d2_timeline_complete(basepath, "PFC")

    
except Exception as e:
    print(f"❌ Failed: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*60)

❌ Failed: No *_CA1_D2_time*.mat under W:\data\PPP\PPP20\day5\complex_system



Traceback (most recent call last):
  File "C:\Users\Cornell\AppData\Local\Temp\ipykernel_70220\39668419.py", line 210, in <module>
    ca1_complete_df, ca1_complete_meta = load_d2_timeline_complete(basepath, "CA1")
                                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Cornell\AppData\Local\Temp\ipykernel_70220\39668419.py", line 67, in load_d2_timeline_complete
    raise FileNotFoundError(f"No *_{region}_D2_time*.mat under {search_dir}")
FileNotFoundError: No *_CA1_D2_time*.mat under W:\data\PPP\PPP20\day5\complex_system


In [16]:
ca1_complete_df

,center_s,d2,d2_err,state
0,54.0,0.161515,NaN,WAKE
1,56.0,0.168827,NaN,WAKE
2,58.0,0.168394,NaN,WAKE
3,60.0,0.176545,NaN,WAKE
4,62.0,0.179839,NaN,WAKE
...,...,...,...,...
10166,29527.0,0.117421,NaN,NREM
10167,29529.0,0.107635,NaN,NREM
10168,29531.0,0.107133,NaN,NREM
10169,29533.0,0.104928,NaN,NREM


In [17]:
epoch_df

,name,startTime,stopTime,environment,manipulation,behavioralParadigm,stimuli,notes,basepath
0,presleep_pulseStim_230302_084139,0.0000,7270.22715,NaN,NaN,NaN,NaN,NaN,X:\data\Barrage\NN13\day23
1,presleep_230302_104425,7270.2272,17019.59035,NaN,NaN,NaN,NaN,NaN,X:\data\Barrage\NN13\day23
2,tmaze_training_230302_132911,17019.5904,18998.13755,NaN,NaN,NaN,NaN,NaN,X:\data\Barrage\NN13\day23
3,postsleep_230302_140330,18998.1376,29567.56475,NaN,NaN,NaN,NaN,NaN,X:\data\Barrage\NN13\day23


In [18]:
# from pathlib import Path
# import pandas as pd, numpy as np
# from pathlib import Path
# import numpy as np, pandas as pd
# from scipy.io import loadmat

# def load_behavior_trials(basepath, basename="day7"):
#     p = Path(basepath) / f"{basename}.animal.behavior.mat"
#     m = loadmat(str(p), squeeze_me=True, struct_as_record=False)  # classic .mat
#     beh = m.get("behavior", None)
#     if beh is None:
#         raise KeyError("No 'behavior' in MAT file")
#     tr = getattr(beh, "trials", None)
#     speed = getattr(beh, "speed_smooth", None)


#     return speed,tr
# speed,trials = load_behavior_trials(basepath, basename=basename)


In [14]:
import numpy as np
from scipy.signal import welch
import nelpy as nel

def _flatten_spike_times(spike_times):
    """Accepts list/tuple/array of arrays or a single 1D array, returns sorted 1D float array."""
    if isinstance(spike_times, (list, tuple)):
        # list of arrays or scalars
        if len(spike_times) == 0:
            return np.array([], dtype=float)
        if hasattr(spike_times[0], "__len__") and not np.isscalar(spike_times[0]):
            times = np.concatenate([np.asarray(st, dtype=float).ravel()
                                    for st in spike_times if st is not None and len(st) > 0])
        else:
            times = np.asarray(spike_times, dtype=float).ravel()
    else:
        arr = np.asarray(spike_times, dtype=object)
        if arr.dtype == object:
            times = np.concatenate([np.asarray(st, dtype=float).ravel()
                                    for st in arr if st is not None and len(st) > 0])
        else:
            times = arr.astype(float).ravel()

    times = times[np.isfinite(times)]
    if times.size == 0:
        raise ValueError("No valid spike times found after flattening.")
    return np.sort(times)

def bin_spikes(spike_times, bin_size=0.04, t_start=None, t_stop=None):
    st = _flatten_spike_times(spike_times)
    if t_start is None: t_start = float(np.floor(st.min()))
    if t_stop  is None: t_stop  = float(np.ceil(st.max()))
    edges = np.arange(t_start, t_stop + bin_size, bin_size)
    counts, _ = np.histogram(st, bins=edges)
    rates = counts / bin_size
    t = edges[:-1] + bin_size/2
    return t, rates

# def segments_from_epochs(t, x, epochs):
#     arr = epochs.data if isinstance(epochs, nel.EpochArray) else np.asarray(epochs)
#     return [x[(t >= s) & (t < e)] for s, e in arr if np.any((t >= s) & (t < e))]

SWR and barrage

In [17]:
import numpy as np
import scipy.io as sio      
import scipy
# Load ripple events
ripple_events = loading.load_ripples_events(basepath=basepath)
ripple_epochs = nel.EpochArray(np.array([ripple_events.start, ripple_events.stop]).T)
ripple_peaks=ripple_events.peaks
    # Load barrage events
ripple_epochs=nrem_epochs&ripple_epochs
try:
    # Load barrage events
    #brpath=r"W:\data\PPP\CA2\PPP7\day8"
    barrage_events = loading.load_barrage_events(basepath=basepath)  
    # Create barrage epochs
    barrage_epochs = nel.EpochArray(np.array([barrage_events.start, barrage_events.stop]).T)
    # Find barrages in NREM epochs
    barrages_in_nrem = barrage_epochs & nrem_epochs
    mat_path = os.path.join(basepath, f'{basename}.HSEn2.events.mat')

    data = scipy.io.loadmat(mat_path, struct_as_record=False, squeeze_me=True)
    barrage = data['HSEn2']
    barrage_timestamps= barrage.timestamps 
    barrage_epochs=nrem_epochs&barrage_epochs
except Exception as e:
    # General exception for unexpected errors
    print(f"No barrage file: {e}")
    barrage_epochs=[]
    # Extract start times

No barrage file: 'DataFrame' object has no attribute 'start'


D:\Documents\GitHub\neuro_py\neuro_py\io\loading.py:1036: UserWarning: No barrage file found for W:\data\PPP\PPP20\day5


d2 vs replay rate

In [ ]:
import scipy.io
import numpy as np
from scipy.ndimage import zoom
from scipy.interpolate import interp1d
import umap
import matplotlib.pyplot as plt

# Load data
mat_path = f'D:/Documents/GitHub/R_Project/PPP/codebook/Criticality_analysis/replay_results/{animal_name}_{day}_replay_sequences.mat'
data = scipy.io.loadmat(mat_path)

# Define epochs and templates
epochs = ['epoch1','epoch3','epoch5']
template = 'template04'  # only use template02
bin_size = 0.01  # 10 ms
target_bins = 30  # fixed number of bins for all events

# Assume these are already loaded:
# E_spikes: list of spike times (one per unit)
# E_CellID: list of unit IDs corresponding to E_spikes

binned_firing_rates = []
avg_pos = []
epoch_labels = []
left_replay = []
right_replay = []
left_replay = []
left_times  = []

right_replay = []
right_times  = []
left_intervals=[]
right_intervals=[]
left_replay_goodfit=[]
right_replay_goodfit=[]
left_replay_slope=[]
right_replay_slope=[]
left_all_bestseq = []
right_all_bestseq = []
left_replay_matrix=[]
right_replay_matrix=[]
for e in epochs:
    key_event = f'{e}_{template}_eventtimes'
    key_seq   = f'{e}_{template}'
    key_pval  = f'{e}_{template}_pvalue'
    key_Goodfit  = f'{e}_{template}_Goodfit'
    key_slope  = f'{e}_{template}_slope'
    key_best_seq  = f'{e}_{template}_all_best_seq'
    if key_event not in data or key_seq not in data:
        continue

    events = data[key_event].squeeze()
    seqs   = data[key_seq][0]
    pvals  = data[key_pval].squeeze()
    Goodfit  = data[key_Goodfit].squeeze()
    slope  = data[key_slope].squeeze()
    best_seq   = data[key_best_seq].squeeze()
    if events.ndim == 1:
        events = np.expand_dims(events, axis=0)

    for i, event in enumerate(events):
        if i >= len(seqs):
            continue

        pval = np.squeeze(pvals[i])   # [left_pval, right_pval]
        s = np.squeeze(seqs[i])       # decoded sequence, shape (T,)
        all_bestseq = np.squeeze(best_seq[i])       # decoded sequence, shape (T,)

        s_half = s.shape[1] // 2
        event_time = np.mean(event)  # use midpoint time

        # Left significant
        if pval[0] < 0.05:
            left_replay.append(np.argmax(s[:.,:s_half], axis=0) )
            left_times.append(event_time)
            left_intervals.append(event)
            left_replay_goodfit.append(Goodfit[i,0])
            left_replay_slope.append(slope[i][0, 0])
            left_all_bestseq.append(all_bestseq[:s_half])
            left_replay_matrix.append(s[:, :s_half])
        # Right significant
        if pval[1] < 0.05:
            right_replay.append(np.argmax(s[:,s_half:], axis=0))
            right_times.append(event_time)
            right_intervals.append(event)
            right_replay_goodfit.append(Goodfit[i,1])
            right_replay_slope.append(slope[i][0, 1])
            right_all_bestseq.append(all_bestseq[s_half:])
            right_replay_matrix.append(s[:, s_half:]) 

SyntaxError: invalid decimal literal (809950681.py, line 1)

replay ripple non-replay ripple

In [ ]:
# === D2 vs Replay Rate Analysis ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr, f_oneway

print("=== D2 vs Replay Rate Analysis ===")

# --- 1. Combine left and right intervals ---
print("Step 1: Combining left and right intervals...")
# (Assuming left_intervals, right_intervals, and ca1_complete_df are pre-loaded)

# Combine all intervals from both left and right
all_intervals = []

if 'left_intervals' in locals() and left_intervals is not None:
    all_intervals.extend(left_intervals)
    print(f"Added {len(left_intervals)} left intervals")

if 'right_intervals' in locals() and right_intervals is not None:
    all_intervals.extend(right_intervals)
    print(f"Added {len(right_intervals)} right intervals")

if len(all_intervals) == 0:
    print("Warning: No intervals found. Make sure left_intervals and right_intervals are defined.")
    all_intervals = [[1000, 1030], [2000, 2030], [3000, 3030]]
    print("Using dummy intervals for demonstration")

print(f"Total combined intervals: {len(all_intervals)}")
combined_intervals = np.array(all_intervals)
print(f"Combined intervals shape: {combined_intervals.shape}")

# --- 2. Filter CA1 data to NREM periods ---
print("\nStep 2: Filtering D2 data to NREM periods...")

# Filter to NREM state
nrem_df = ca1_complete_df[ca1_complete_df["state"] == "NREM"].copy()
print(f"NREM data points: {len(nrem_df)}")

# Extract center times and d2 values
centers = nrem_df["center_s"].values
d2_values = nrem_df["d2"].values

# === MODIFICATION: Check for d2_err column ===
if "d2_err" in nrem_df.columns:
    d2_errors = nrem_df["d2_err"].values
    has_d2err = True
    print("Found 'd2_err' column for error propagation.")
else:
    d2_errors = np.full_like(d2_values, np.nan) # Fill with NaN
    has_d2err = False
    print("Warning: 'd2_err' column not found. Shaded band will not be plotted.")
# === END MODIFICATION ===

print(f"NREM time range: {centers.min():.1f} - {centers.max():.1f} seconds")
print(f"D2 value range: {d2_values.min():.4f} - {d2_values.max():.4f}")

# --- 3. Calculate replay rates for D2 windows ---
print("\nStep 3: Calculating replay rates for D2 time windows...")

d2_window_size = 30.0  # 30 second windows
d2_step_size = 2.0     # 2 second sliding window

replay_rates = []

# Simplified loop
for i, center_time in enumerate(centers):
    # Define the 30-second window around this D2 measurement
    window_start = center_time - d2_window_size / 2
    window_end = center_time + d2_window_size / 2
    
    # Count replay events (intervals) that overlap with this window
    replay_count = 0
    for interval_start, interval_end in combined_intervals:
        # Check if interval overlaps with D2 window
        if (interval_start < window_end) and (interval_end > window_start):
            replay_count += 1
    
    # Calculate replay rate (events per second)
    replay_rate = replay_count / d2_window_size
    replay_rates.append(replay_rate)

replay_rates = np.array(replay_rates)
print(f"Calculated replay rates for {len(replay_rates)} D2 measurements")
print(f"Replay rate range: {replay_rates.min():.4f} - {replay_rates.max():.4f} events/sec")

In [ ]:
# === Separate equal-width 6-bin tiling per group (no pooling, no normalization) ===
# --- 3b. Ripple epochs: overlap vs non-overlap with replay intervals ---
print("\nStep 3b: Processing ripple epochs...")

def _to_intervals(obj):
    """
    Accepts several 'ripple_epoch' formats and returns float ndarray (N,2)
    Accepts:
      - numpy array shape (N,2)
      - dict-like with key 'data'
      - pandas DataFrame with ['start','end'] or first two columns
    """
    import numpy as _np
    import pandas as _pd

    if obj is None:
        return _np.empty((0, 2), float)

    if isinstance(obj, _np.ndarray):
        arr = obj
    elif isinstance(obj, dict) and 'data' in obj:
        arr = _np.asarray(obj['data'])
    elif isinstance(obj, _pd.DataFrame):
        if {'start','end'}.issubset(set(obj.columns)):
            arr = obj[['start','end']].to_numpy()
        else:
            arr = obj.iloc[:, :2].to_numpy()
    else:
        # last resort: try to numpy-ify
        arr = _np.asarray(obj)

    arr = arr.astype(float)
    arr = arr[_np.isfinite(arr).all(1)]
    arr = arr[arr[:,0] < arr[:,1]]
    return arr

# Try a few common variable names for ripple epochs
ripple_intervals = None
for candidate in ('ripple_epoch', 'ripple_epochs', 'ripple_epoch_data', 'ripples'):
    if candidate in locals():
        ripple_intervals = locals()[candidate]
        break

ripples = ripple_epochs.data
print(f"Total ripple intervals: {len(ripples)}")

# Utility: interval overlap (vectorized)
def overlaps_any(interval, others):
    a0, a1 = interval
    if others.size == 0:
        return False
    # (start < other_end) & (end > other_start)
    return np.any((a0 < others[:,1]) & (a1 > others[:,0]))

# Build masks for ripples that overlap ANY replay interval vs not
if combined_intervals.size == 0:
    print("Warning: No replay intervals available; treating all ripples as non-overlap.")
    ripples_overlap_mask = np.zeros(len(ripples), dtype=bool)
else:
    ripples_overlap_mask = np.fromiter(
        (overlaps_any(r, combined_intervals) for r in ripples),
        dtype=bool, count=len(ripples)
    )

ripples_in_replay     = ripples[ripples_overlap_mask]
ripples_outside_replay = ripples[~ripples_overlap_mask]

print(f"Ripples overlapping replay intervals: {len(ripples_in_replay)}")
print(f"Ripples outside replay intervals:   {len(ripples_outside_replay)}")

# --- 3c. Compute per-D²-window rates for:
#         (i) replay intervals
#         (ii) ALL ripples
#         (iii) ripples overlapping replay
#         (iv) ripples outside replay
def count_overlaps_in_window(intervals, w0, w1):
    if intervals.size == 0:
        return 0
    return int(np.sum((intervals[:,0] < w1) & (intervals[:,1] > w0)))

replay_rates = []
ripple_rate_all = []
ripple_rate_in  = []
ripple_rate_out = []
valid_indices   = []

for i, (center_time, d2_val) in enumerate(zip(centers, d2_values)):
    w0 = center_time - d2_window_size/2.0
    w1 = center_time + d2_window_size/2.0

    # Counts
    replay_cnt = count_overlaps_in_window(combined_intervals, w0, w1)
    r_all_cnt  = count_overlaps_in_window(ripples,               w0, w1)
    r_in_cnt   = count_overlaps_in_window(ripples_in_replay,     w0, w1)
    r_out_cnt  = count_overlaps_in_window(ripples_outside_replay,w0, w1)

    # Rates (events/sec)
    replay_rates.append(replay_cnt / d2_window_size)
    ripple_rate_all.append(r_all_cnt / d2_window_size)
    ripple_rate_in.append(r_in_cnt / d2_window_size)
    ripple_rate_out.append(r_out_cnt / d2_window_size)
    valid_indices.append(i)

replay_rates     = np.asarray(replay_rates, float)
ripple_rate_all  = np.asarray(ripple_rate_all, float)
ripple_rate_in   = np.asarray(ripple_rate_in,  float)
ripple_rate_out  = np.asarray(ripple_rate_out, float)

print(f"Replay rate range: {replay_rates.min():.4f} - {replay_rates.max():.4f} events/sec")
print(f"Ripple(all) rate range: {ripple_rate_all.min():.4f} - {ripple_rate_all.max():.4f}")
print(f"Ripple(in)  rate range: {ripple_rate_in.min():.4f}  - {ripple_rate_in.max():.4f}")
print(f"Ripple(out) rate range: {ripple_rate_out.min():.4f} - {ripple_rate_out.max():.4f}")

# --- 4. Extend analysis_df with ripple rates ---
analysis_df = pd.DataFrame({
    'center_time':      centers[valid_indices],
    'd2_value':         d2_values[valid_indices],
    'd2_6tile':         d2_values[valid_indices],
    'd2_err':           d2_errors[valid_indices],
    'replay_rate':      replay_rates,
    'ripple_rate_all':  ripple_rate_all,
    'ripple_rate_in':   ripple_rate_in,
    'ripple_rate_out':  ripple_rate_out,
})
print(f"\nExtended analysis dataframe shape: {analysis_df.shape}")
print(analysis_df.head(3))
print("\nStep 6: 6-tile binned analysis for replay & ripple rates...")

clean_df = analysis_df.dropna(subset=['d2_6tile', 'replay_rate', 'ripple_rate_all', 'ripple_rate_in', 'ripple_rate_out']).copy()

df = clean_df.dropna(subset=['d2_6tile','ripple_rate_in','ripple_rate_out']).copy()
ycol = 'd2_6tile'
n_bins = 6
labels6 = [f'Q{i}' for i in range(1, n_bins+1)]

def build_equal_edges(series, n_bins=6):
    vals = np.asarray(series, float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return None
    vmin, vmax = np.min(vals), np.max(vals)
    if vmin == vmax:
        # expand slightly so pd.cut gets 2 distinct edges
        eps = 1e-9 if vmin == 0 else 1e-6 * abs(vmin)
        vmin -= eps; vmax += eps
    return np.linspace(vmin, vmax, n_bins + 1)

def summarize_by_edges(df_in, rate_col, edges, ycol='d2_6tile', labels=None):
    if edges is None:
        return pd.DataFrame(columns=['mean','sem','count'])
    d = df_in[[rate_col, ycol]].dropna().copy()
    d['_tile'] = pd.cut(d[rate_col], bins=edges, labels=labels, include_lowest=True, right=True)
    g = d.groupby('_tile', observed=True)[ycol]
    out = pd.DataFrame({'mean': g.mean(), 'sem': g.sem(), 'count': g.count()})
    return out.reindex(labels) if labels is not None else out

# Build edges per group (SEPARATELY)
edges_in  = build_equal_edges(df['ripple_rate_in'],  n_bins=n_bins)
edges_out = build_equal_edges(df['ripple_rate_out'], n_bins=n_bins)

# Summaries (each uses its own edges)
summ_in  = summarize_by_edges(df, 'ripple_rate_in',  edges_in,  ycol=ycol, labels=labels6)
summ_out = summarize_by_edges(df, 'ripple_rate_out', edges_out, ycol=ycol, labels=labels6)

# Print the bin edges per tile
print("IN  ripple-rate bin edges:", edges_in)
print("OUT ripple-rate bin edges:", edges_out)
print("IN counts per tile:",  summ_in['count'].fillna(0).astype(int).tolist())
print("OUT counts per tile:", summ_out['count'].fillna(0).astype(int).tolist())

In [ ]:

from pathlib import Path
import pandas as pd

# Inputs:
# ca1_complete_df : pandas DataFrame
# animal_name     : str
# day             : int | str
# brain_region    : str

out_dir = Path(r"D:\Documents\GitHub\Sleep_criticaltiy\data\ripple_barr\replay_non_replay")
out_dir.mkdir(parents=True, exist_ok=True)

fname = f"{animal_name}_{day}_{brain_region}.pkl"
save_path = out_dir / fname

analysis_df.to_pickle(save_path)
print(f"Saved to: {save_path}")

